# Qwen 7B corrected repeated-sampling development run

1. Select an **L4 GPU** in Colab.
2. Upload the INPUT `qwen7b_corrected_repeated_transfer_v1.zip` in the upload cell.
   Renamed uploads are accepted only when their bytes match the approved SHA256.
3. Run the execution cell **once**. It installs the isolated environment, validates
   the pinned plan, checks every prompt, starts vLLM, sends 300 sequential requests,
   replays evidence, stops the server and exports a new evidence ZIP.
4. Run the DOWNLOAD cell. Save the OUTPUT evidence ZIP to your computer and attach
   it in our chat. Do not upload the output into the INPUT cell. After saving,
   disconnect and delete the Colab GPU runtime.

This explicitly executes a new development batch: 30 corrected cases, 15 groups,
15 clean/perturbed pairs, ten independent one-choice samples per case. Preparation
manifests remain unchanged with dispatch_authorized=false; running this reviewed
execution cell supplies the separate operator authorization for this batch only.
It does not enable held-out evaluation or a final protocol freeze.

Settings: Qwen/Qwen2.5-7B-Instruct revision
`a09a35458c702b33eeacc393d103063234e8bc28`, BF16, vLLM 0.30.0,
Transformers 5.20.2, default pinned tokenizer template, context 8192,
GPU utilization 0.88, temperature 0.7, top_p 1.0, output cap 1024,
logprobs/top alternatives 5, seeds 1729-1738, n=1. The installed sampling
constructor must retain the requested temperature, top_p and per-slot seed.
Other inherited model defaults are recorded. Loaded revisions are operator
attested; /models verifies model ID, not weight bytes.

Repository: `0bf4fe6a8069397525f42af3bf5579b977fdd05f`.
Transfer: `1e69fd27c4f392bf540189afe572b6f229098fe98076753c03d27b2f0968bfdd`.
Manifest: `0468034b1a302dfba56b294e69d09fe68aaa85132c6aa3069d61eae0a3e565ba`.

Each sample has a durable intent written before its HTTP request. There are
**zero automatic retries**, including errors and unknown outcomes. This is an
explicit safety amendment to the historical diagnostic's retry-on-resume text;
probability-free clustering semantics remain frozen. This notebook refuses an
existing execution or run folder. After interruption, preserve files and request
review; do not reset and resend the batch. Partial evidence is exported when the
Python process can run its cleanup; forced runtime deletion can lose evidence.

This notebook saves individual raw responses, parser/token-evidence replay,
usage, timing and finish status. Official correctness, repeated entropy and
disagreement, and matched single-sample comparisons are later CPU analyses.
No reference sidecars or labels are generation inputs. Prompt text may contain
benchmark credential literals. No environment or authentication token is exported.
No user HF token is required for this public model.

The isolated setup may take several minutes; model loading/compilation is separate
from generation. Progress appears every 30 seconds during dispatch. There is no
quota or completion-time guarantee. Session/request wall time is not GPU active
time, full allocation cost, or a controlled model-speed comparison.


## 1. Upload the approved transfer ZIP
Select corrected_qwen7b_coverage_transfer_v1.zip from your local artifacts folder.


In [ ]:
from pathlib import Path
import hashlib
from google.colab import files

expected = '1e69fd27c4f392bf540189afe572b6f229098fe98076753c03d27b2f0968bfdd'
target = Path('/content/qwen7b_corrected_repeated_transfer_v1.zip')
if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected:
    print('Approved INPUT transfer already available; no upload needed.')
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Select one INPUT transfer ZIP.')
    raw = next(iter(uploaded.values()))
    if hashlib.sha256(raw).hexdigest() != expected:
        raise RuntimeError('Wrong transfer bytes. Select qwen7b_corrected_repeated_transfer_v1.zip.')
    if target.exists():
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise RuntimeError('Canonical input differs; preserve it for review.')
    else:
        with target.open('xb') as destination:
            destination.write(raw)
    print('INPUT transfer verified. Run the execution cell once.')


## Execute once: setup, 300 requests, replay and export


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

TASK = Path('/content/uqroute_qwen7b_repeated_execution_v1')
if TASK.exists():
    raise RuntimeError('Execution folder already exists. Preserve it; do not rerun this task or unknown requests.')
source = Path('/content/qwen7b_corrected_repeated_transfer_v1.zip')
expected = '1e69fd27c4f392bf540189afe572b6f229098fe98076753c03d27b2f0968bfdd'
raw = source.read_bytes()
if hashlib.sha256(raw).hexdigest() != expected:
    raise RuntimeError('Transfer ZIP fingerprint differs; nothing started.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True)
print('Allocated GPU:', gpu.stdout.strip(), flush=True)
if 'L4' not in gpu.stdout:
    raise RuntimeError('Select an L4 GPU for this notebook before starting.')
TASK.mkdir(exist_ok=False)
(TASK / 'transfer.zip').write_bytes(raw)
ENV = TASK / 'venv'
PYTHON = ENV / 'bin/python'
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(ENV)], check=True)
print('Installing isolated dependencies. This can take several minutes.', flush=True)
with (TASK / 'environment_install.log').open('xb') as log:
    install = subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install', 'pip', 'vllm==0.30.0', 'transformers==5.20.2', 'ninja'], stdout=log, stderr=subprocess.STDOUT)
if install.returncode:
    raise RuntimeError('Dependency installation failed. Preserve execution folder and environment_install.log.')
env = os.environ.copy()
env['PATH'] = str(PYTHON.parent) + os.pathsep + env.get('PATH', '')
env['VIRTUAL_ENV'] = str(ENV)
env.pop('PYTHONPATH', None)
with (TASK / 'dependency_check.log').open('xb') as log:
    checked = subprocess.run([str(PYTHON), '-m', 'pip', 'check'], env=env, stdout=log, stderr=subprocess.STDOUT)
if checked.returncode:
    raise RuntimeError('Dependency validation failed; preserve logs.')

CLIENT_SOURCE = '"""Repeated-slot durable dispatch and evidence replay. Never retry occupied slots."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport urllib.request\nimport urllib.error\n\nMANIFEST=\'0468034b1a302dfba56b294e69d09fe68aaa85132c6aa3069d61eae0a3e565ba\'\nMODEL=\'Qwen/Qwen2.5-7B-Instruct\'\nREV=\'a09a35458c702b33eeacc393d103063234e8bc28\'\n\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef canonical(v):return json.dumps(v,sort_keys=True,separators=(\',\',\':\'),ensure_ascii=False)\ndef save(path,raw):\n    with path.open(\'xb\') as f:f.write(raw);f.flush();os.fsync(f.fileno())\ndef write(path,value):save(path,(canonical(value)+\'\\n\').encode())\ndef safe(path):\n    if any(p.is_symlink() for p in (path,*path.parents)):raise ValueError(\'Symlink rejected\')\n\n\ndef load(prepared):\n    safe(prepared);raw=(prepared/\'manifest.json\').read_bytes()\n    if sha(raw)!=MANIFEST:raise ValueError(\'Manifest differs\')\n    m=json.loads(raw);data=(prepared/\'requests.json\').read_bytes();rows=json.loads(data)\n    if sha(data)!=m[\'requests_sha256\'] or len(rows)!=300 or m[\'held_out_access_enabled\'] is not False or m[\'main_experiment_ready\'] is not False or m[\'scope\']!=\'corrected-primary-development-repeated-sampling-proposal\':raise ValueError(\'Plan scope differs\')\n    if m[\'model_settings\'][\'model_id\']!=MODEL or any(m[\'model_settings\'][k]!=REV for k in (\'model_revision\',\'tokenizer_revision\')):raise ValueError(\'Model differs\')\n    grouped={}\n    for r in rows:\n        i=r[\'sample_index\'];b=r[\'request\']\n        if type(i) is not int or i not in range(10):raise ValueError(\'Sample index\')\n        if r[\'slot_id\']!=sha(canonical([r[\'case_key\'],i]).encode()) or r[\'request_sha256\']!=sha(canonical(b).encode()):raise ValueError(\'Slot/request hash differs\')\n        if set(b)!={\'model\',\'messages\',\'temperature\',\'top_p\',\'max_tokens\',\'logprobs\',\'top_logprobs\',\'stream\',\'seed\',\'n\'}:raise ValueError(\'Request fields differ\')\n        expected={\'model\':MODEL,\'temperature\':.7,\'top_p\':1.,\'max_tokens\':1024,\'logprobs\':True,\'top_logprobs\':5,\'stream\':False,\'seed\':1729+i,\'n\':1}\n        if any(b[k]!=v for k,v in expected.items()) or [x[\'role\'] for x in b[\'messages\']]!=[\'system\',\'user\']:raise ValueError(\'Request settings differ\')\n        group=grouped.setdefault(r[\'case_key\'],[]);group.append(r)\n    if list(grouped)!=m[\'case_keys\'] or len(grouped)!=30 or len({r[\'slot_id\'] for r in rows})!=300:raise ValueError(\'Case set differs\')\n    for values in grouped.values():\n        if [r[\'sample_index\'] for r in values]!=list(range(10)):raise ValueError(\'Ten ordered slots required\')\n        base=values[0]\n        if any(any(r[f]!=base[f] for f in (\'benchmark\',\'canonical_base_id\',\'source_file\')) or r[\'request\'][\'messages\']!=base[\'request\'][\'messages\'] for r in values):raise ValueError(\'Slot prompt/lineage differs\')\n    return m,rows\n\n\ndef transport(body):\n    request=urllib.request.Request(\'http://127.0.0.1:8000/v1/chat/completions\',data=canonical(body).encode(),headers={\'Content-Type\':\'application/json\'})\n    try:\n        with urllib.request.urlopen(request,timeout=600) as response:return response.status,response.read()\n    except urllib.error.HTTPError as e:return e.code,e.read()\n\n\ndef dispatch(rows,output,binding,caller=transport):\n    safe(output);output.mkdir(exist_ok=False)\n    write(output/\'run_manifest.json\',binding);(output/\'cases\').mkdir()\n    start=time.monotonic()\n    for index,row in enumerate(rows):\n        folder=output/\'cases\'/row[\'slot_id\'];folder.mkdir(exist_ok=False)\n        write(folder/\'intent.json\',{\'case_key\':row[\'case_key\'],\'sample_index\':row[\'sample_index\'],\'slot_id\':row[\'slot_id\'],\'request_sha256\':row[\'request_sha256\'],\'created_unix\':time.time()})\n        before=time.monotonic()\n        try:status,raw=caller(row[\'request\'])\n        except BaseException as e:\n            write(folder/\'transport_unknown.json\',{\'exception_type\':type(e).__name__,\'elapsed_seconds\':time.monotonic()-before,\'outcome\':\'unknown_do_not_retry\'})\n            raise\n        save(folder/\'http_response.bin\',raw)\n        write(folder/\'response_metadata.json\',{\'http_status\':status,\'elapsed_seconds\':time.monotonic()-before,\'response_sha256\':sha(raw)})\n        if (index+1)%10==0:print(f\'Repeated slots saved: {index+1}/{len(rows)}\',flush=True)\n    write(output/\'session.json\',{\'processed\':len(rows),\'batch_wall_seconds\':time.monotonic()-start,\'automatic_retries\':0})\n\n\ndef replay(rows,run_dir,output,binding,evaluate):\n    safe(run_dir);safe(output)\n    if output.exists():raise ValueError(\'Replay exists\')\n    if json.loads((run_dir/\'run_manifest.json\').read_bytes())!=binding:raise ValueError(\'Run binding differs\')\n    expected={r[\'slot_id\'] for r in rows}\n    if any(p.name not in expected or p.is_symlink() for p in (run_dir/\'cases\').iterdir()):raise ValueError(\'Unexpected slot directory\')\n    records=[];hashes={};totals=Counter();missing=Counter()\n    for row in rows:\n        folder=run_dir/\'cases\'/row[\'slot_id\'];safe(folder)\n        record={k:row[k] for k in (\'case_key\',\'sample_index\',\'slot_id\',\'benchmark\',\'canonical_base_id\',\'source_file\')}\n        intent=folder/\'intent.json\';response=folder/\'http_response.bin\';metadata=folder/\'response_metadata.json\'\n        if not intent.exists():record[\'status\']=\'unattempted\'\n        else:\n            value=json.loads(intent.read_bytes())\n            if any(value[k]!=row[k] for k in (\'case_key\',\'sample_index\',\'slot_id\',\'request_sha256\')):raise ValueError(\'Intent differs\')\n            if not response.exists() or not metadata.exists():record[\'status\']=\'unknown_outcome\'\n            else:\n                raw=response.read_bytes();meta=json.loads(metadata.read_bytes())\n                if meta[\'response_sha256\']!=sha(raw):raise ValueError(\'Response hash differs\')\n                record[\'http_status\']=meta[\'http_status\'];record[\'elapsed_seconds\']=meta[\'elapsed_seconds\']\n                try:value=json.loads(raw)\n                except (ValueError,UnicodeDecodeError):value=None\n                if meta[\'http_status\']!=200:record[\'status\']=\'http_error\'\n                elif not isinstance(value,dict):record[\'status\']=\'malformed_response\'\n                else:record.update(evaluate(value,row[\'benchmark\']))\n                usage=value.get(\'usage\') if isinstance(value,dict) else None\n                for key in (\'prompt_tokens\',\'completion_tokens\',\'total_tokens\'):\n                    n=usage.get(key) if isinstance(usage,dict) else None\n                    if type(n) is int and n>=0:totals[key]+=n\n                    else:missing[key]+=1\n        records.append(record)\n    for p in run_dir.rglob(\'*\'):\n        safe(p)\n        if p.is_file():hashes[p.relative_to(run_dir).as_posix()]=sha(p.read_bytes())\n    summary={\'rows\':len(rows),\'cases\':30,\'status_counts\':dict(Counter(r[\'status\'] for r in records)),\'whole_output_eligible\':sum(r.get(\'whole_output_scores\') is not None for r in records),\'meaningful_token_eligible\':sum(r.get(\'meaningful_token_score\') is not None for r in records),\'known_token_totals\':dict(totals),\'missing_usage_returned_slots\':dict(missing),\'held_out_access_enabled\':False,\'official_scoring\':\'not_performed\',\'automatic_retries\':0,\'prior_files_modified\':False,\'outcome_file_sha256\':hashes}\n    output.mkdir(exist_ok=False);write(output/\'summary.json\',summary);write(output/\'records.json\',records)\n    print(canonical({k:v for k,v in summary.items() if k!=\'outcome_file_sha256\'}),flush=True)\n    return summary\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=[\'run\',\'replay\']);p.add_argument(\'--task\',type=Path,required=True);a=p.parse_args()\n    task=a.task.resolve();repo=task/\'repo\';sys.path.insert(0,str(repo/\'tools/development_audits/corrected_generation_v1\'));import workflow\n    manifest,rows=load(task/\'prepared\');runtime_raw=(task/\'metadata/runtime.json\').read_bytes();runtime=json.loads(runtime_raw);workflow.check_runtime(runtime,manifest)\n    binding={\'version\':\'uqroute-corrected-repeated-execution-1.0.0\',\'prepared_manifest_sha256\':MANIFEST,\'runtime_sha256\':sha(runtime_raw),\'client_sha256\':sha(Path(__file__).read_bytes()),\'request_count\':300,\'automatic_retries\':0,\'held_out_access_enabled\':False,\'main_experiment_ready\':False,\'retry_amendment\':\'Fresh corrected run: no automatic retries, including terminal errors or unknown outcomes; historical Task 2 resume retry text is not applied.\'}\n    artifact=repo/\'artifacts/local_development_audits\';artifact.mkdir(parents=True,exist_ok=True)\n    run_dir=artifact/\'corrected_qwen7b_repeated_run_v1\'\n    if a.mode==\'run\':dispatch(rows,run_dir,binding)\n    else:\n        frozen=workflow.methods(repo)\n        replay(rows,run_dir,artifact/\'corrected_qwen7b_repeated_replay_v1\',binding,lambda response,bench:workflow.evaluate(response,bench,frozen))\n\n\nif __name__==\'__main__\':main()\n'
(TASK / 'repeated_client.py').write_text(CLIENT_SOURCE, encoding='utf-8')
RUNNER_SOURCE = '"""Pinned corrected Qwen 7B repeated development; no automatic request retries."""\nfrom pathlib import Path\nfrom importlib import metadata\nfrom dataclasses import asdict\nimport hashlib\nimport json\nimport os\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport uuid\nimport zipfile\n\nMODEL=\'Qwen/Qwen2.5-7B-Instruct\'\nREV=\'a09a35458c702b33eeacc393d103063234e8bc28\'\nCODE=\'0bf4fe6a8069397525f42af3bf5579b977fdd05f\'\nMANIFEST=\'0468034b1a302dfba56b294e69d09fe68aaa85132c6aa3069d61eae0a3e565ba\'\nZIP_SHA=\'1e69fd27c4f392bf540189afe572b6f229098fe98076753c03d27b2f0968bfdd\'\nTASK=Path(sys.argv[1]).resolve()\nMETA=TASK/\'metadata\';META.mkdir(exist_ok=False)\nPREP=TASK/\'prepared\';PREP.mkdir(exist_ok=False)\nREPO=TASK/\'repo\'\nSERVER=None\nGENERATION=None\nfailure=None\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef write(name,value):\n    with (META/name).open(\'x\',encoding=\'utf-8\') as f:json.dump(value,f,indent=2);f.write(\'\\n\')\ndef cmd(args,**kw):return subprocess.run(args,check=True,**kw)\ndef stop():\n    if GENERATION is not None and GENERATION.poll() is None:\n        os.killpg(GENERATION.pid,signal.SIGTERM)\n        try:GENERATION.wait(timeout=15)\n        except subprocess.TimeoutExpired:\n            os.killpg(GENERATION.pid,signal.SIGKILL);GENERATION.wait(timeout=15)\n    if SERVER is not None and SERVER.poll() is None:\n        os.killpg(SERVER.pid,signal.SIGTERM)\n        try:SERVER.wait(timeout=30)\n        except subprocess.TimeoutExpired:\n            os.killpg(SERVER.pid,signal.SIGKILL);SERVER.wait(timeout=15)\n    write(\'shutdown.json\',{\'server_stopped\':SERVER is None or SERVER.poll() is not None,\'held_out_access_enabled\':False})\n\ntry:\n    import torch\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():raise RuntimeError(\'CUDA and BF16 are required; do not change frozen dtype.\')\n    device=torch.cuda.get_device_properties(0)\n    if \'L4\' not in device.name:raise RuntimeError(\'This notebook is prepared for L4; select L4 before starting.\')\n    envreport={\'python\':sys.version,\'torch\':metadata.version(\'torch\'),\'vllm\':metadata.version(\'vllm\'),\n               \'transformers\':metadata.version(\'transformers\'),\'torch_cuda_build\':torch.version.cuda,\n               \'gpu_name\':device.name,\'gpu_memory_mib\':device.total_memory//1024**2,\n               \'bfloat16_supported\':True,\'held_out_access_enabled\':False}\n    if envreport[\'vllm\']!=\'0.30.0\' or envreport[\'transformers\']!=\'5.20.2\':raise RuntimeError(\'Dependency versions differ\')\n    write(\'environment.json\',envreport)\n    ninja=shutil.which(\'ninja\')\n    if not ninja:raise RuntimeError(\'Ninja executable absent from PATH.\')\n    write(\'ninja_check.json\',{\'path\':ninja,\'version\':cmd([ninja,\'--version\'],capture_output=True,text=True).stdout.strip()})\n    print(\'Environment verified; validating transfer.\',flush=True)\n    archive=TASK/\'transfer.zip\'\n    if sha(archive.read_bytes())!=ZIP_SHA:raise RuntimeError(\'Transfer hash differs.\')\n    with zipfile.ZipFile(archive) as z:\n        if sorted(z.namelist())!=[\'manifest.json\',\'requests.json\'] or z.testzip() is not None:raise RuntimeError(\'Unexpected or corrupt transfer.\')\n        payloads={n:z.read(n) for n in z.namelist()}\n    if sha(payloads[\'manifest.json\'])!=MANIFEST:raise RuntimeError(\'Manifest hash differs.\')\n    for n,raw in payloads.items():\n        with (PREP/n).open(\'xb\') as f:f.write(raw)\n    cmd([\'git\',\'clone\',\'--no-checkout\',\'https://github.com/RavindraSSK/uqroute-tc.git\',str(REPO)],capture_output=True,text=True)\n    cmd([\'git\',\'-C\',str(REPO),\'checkout\',\'--detach\',CODE],capture_output=True,text=True)\n    head=cmd([\'git\',\'-C\',str(REPO),\'rev-parse\',\'HEAD\'],capture_output=True,text=True).stdout.strip()\n    if head!=CODE or cmd([\'git\',\'-C\',str(REPO),\'status\',\'--porcelain\'],capture_output=True,text=True).stdout:raise RuntimeError(\'Repository identity differs.\')\n    sys.path.insert(0,str(REPO/\'tools/development_audits/corrected_generation_v1\'))\n    import workflow\n    sys.path.insert(0,str(TASK))\n    import repeated_client\n    manifest,requests=repeated_client.load(PREP)\n    if len(requests)!=300 or manifest[\'scope\']!=\'corrected-primary-development-repeated-sampling-proposal\' or manifest[\'model_settings\'][\'model_id\']!=MODEL:raise RuntimeError(\'Wrong plan.\')\n    for name in (\'model_revision\',\'tokenizer_revision\'):\n        if manifest[\'model_settings\'][name]!=REV:raise RuntimeError(\'Model revision differs.\')\n    write(\'repository_validation.json\',{\'revision\':head,\'manifest_sha256\':MANIFEST,\'requests\':len(requests),\'held_out_access_enabled\':False})\n    from transformers import AutoTokenizer\n    from huggingface_hub import hf_hub_download\n    tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REV,trust_remote_code=False)\n    counts=[];over=[]\n    for r in requests:\n        ids=tokenizer.apply_chat_template(r[\'request\'][\'messages\'],tokenize=True,add_generation_prompt=True,return_dict=False)\n        if not isinstance(ids,list) or not ids or not all(type(x) is int for x in ids):raise RuntimeError(\'Unexpected tokenizer IDs.\')\n        counts.append(len(ids))\n        if len(ids)+r[\'request\'][\'max_tokens\']>8192:over.append(r[\'case_key\'])\n    write(\'tokenizer_preflight.json\',{\'requests\':len(counts),\'prompt_tokens_total\':sum(counts),\'prompt_tokens_min\':min(counts),\'prompt_tokens_max\':max(counts),\'output_allowance\':1024,\'context_limit\':8192,\'overflow_cases\':over,\'per_case_prompt_tokens\':[{\'case_key\':r[\'case_key\'],\'prompt_tokens\':count} for r,count in zip(requests,counts)],\'tokenizer_revision\':REV,\'held_out_access_enabled\':False})\n    if over:raise RuntimeError(\'Context overflow; no truncation or case dropping allowed.\')\n    print(f\'All 300 prompts fit: maximum {max(counts)} + 1024 <= 8192.\',flush=True)\n    generation_raw=Path(hf_hub_download(MODEL,\'generation_config.json\',revision=REV)).read_bytes()\n    with (META/\'model_generation_config.json\').open(\'xb\') as f:f.write(generation_raw)\n    defaults=json.loads(generation_raw)\n    from vllm.entrypoints.openai.chat_completion.protocol import ChatCompletionRequest\n    observed=set()\n    for r in requests:\n        p=ChatCompletionRequest(**r[\'request\']).to_sampling_params(max_tokens=r[\'request\'][\'max_tokens\'],default_sampling_params=defaults)\n        if p.temperature!=.7 or p.top_p!=1.0 or p.seed!=r[\'request\'][\'seed\']:raise RuntimeError(\'Frozen repeated sampling differs.\')\n        observed.add((p.temperature,p.top_p,p.top_k,p.repetition_penalty,p.max_tokens))\n    if len(observed)!=1:raise RuntimeError(\'Nonuniform constructor settings; inspect evidence.\')\n    temperature,top_p,top_k,repetition,max_tokens=next(iter(observed))\n    write(\'sampling_constructor_check.json\',{\'requested_temperature\':0.7,\'constructed_temperature\':temperature,\'top_p\':top_p,\'top_k\':top_k,\'repetition_penalty\':repetition,\'max_tokens\':max_tokens,\'requests\':300,\'scope\':\'Local installed constructor using pinned model generation configuration; not per-request server observation.\'})\n    artifact=REPO/\'artifacts/local_development_audits\';artifact.mkdir(parents=True,exist_ok=True)\n    RUN=artifact/\'corrected_qwen7b_repeated_run_v1\';REPLAY=artifact/\'corrected_qwen7b_repeated_replay_v1\'\n    if RUN.exists() or REPLAY.exists():raise RuntimeError(\'Existing run/replay; preserve it.\')\n    with socket.socket() as probe:\n        if probe.connect_ex((\'127.0.0.1\',8000))==0:raise RuntimeError(\'Port 8000 occupied; preserve existing server.\')\n    command=[sys.executable,\'-m\',\'vllm.entrypoints.openai.api_server\',\'--model\',MODEL,\'--revision\',REV,\'--tokenizer-revision\',REV,\'--dtype\',\'bfloat16\',\'--max-model-len\',\'8192\',\'--gpu-memory-utilization\',\'0.88\',\'--host\',\'127.0.0.1\',\'--port\',\'8000\']\n    write(\'launch_argv.json\',command)\n    with (META/\'server.log\').open(\'xb\') as log:SERVER=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n    start=time.monotonic();last=0\n    while True:\n        if SERVER.poll() is not None:raise RuntimeError(\'Server exited during startup; inspect metadata/server.log.\')\n        if time.monotonic()-start>1200:raise RuntimeError(\'Server readiness deadline exceeded; preserve logs.\')\n        try:\n            with urllib.request.urlopen(\'http://127.0.0.1:8000/v1/models\',timeout=5) as response:models=json.load(response)\n            if MODEL not in {r[\'id\'] for r in models.get(\'data\',[])}:raise RuntimeError(\'Wrong served model.\')\n            break\n        except (OSError,TimeoutError):\n            if time.monotonic()-last>45:print(\'Server loading/compiling; waiting automatically.\',flush=True);last=time.monotonic()\n            time.sleep(2)\n    snapshot=(META/\'server.log\').read_bytes()\n    with (META/\'server_before_generation.log\').open(\'xb\') as f:f.write(snapshot)\n    runtime={k:manifest[\'model_settings\'][k] for k in (\'model_id\',\'model_revision\',\'tokenizer_revision\',\'dtype\',\'quantization\')}\n    runtime.update(vllm_version=metadata.version(\'vllm\'),max_model_len=8192,gpu_memory_utilization=0.88,\n                   chat_template=manifest[\'server_settings\'][\'chat_template\'],effective_temperature=temperature,\n                   gpu_name=device.name,gpu_memory_mib=device.total_memory//1024**2,platform=\'Google Colab L4\',\n                   quota_note=\'Remaining provider quota not measured; setup/loading/idle allocation separate.\',\n                   launch_command_sha256=sha((META/\'launch_argv.json\').read_bytes()),server_log_sha256=sha(snapshot))\n    workflow.check_runtime(runtime,manifest);write(\'runtime.json\',runtime)\n    write(\'runtime_notes.json\',{\'effective_temperature_evidence\':\'Local constructor observation, not independent per-request server sampling observation.\',\'server_config\':\'Default pinned model generation config; no override.\',\'generation_config_sha256\':sha(generation_raw),\'server_identity_limit\':\'Revisions operator-attested; models endpoint checks ID only.\',\'held_out_access_enabled\':False})\n    write(\'execution_authorization.json\',{\'scope\':\'User-run corrected development notebook, 300 fresh slots\',\'preparation_manifest_dispatch_authorized\':False,\'execution_action\':\'Explicit execution cell authorizes this new development batch only\',\'retry_amendment\':\'No retries for any occupied or unknown slot; differs from historical repeated diagnostic resume policy.\',\'held_out_access_enabled\':False})\n    print(\'Server ready. Executing the 300 development requests once; no automatic retries.\',flush=True)\n    command=[sys.executable,str(TASK/\'repeated_client.py\'),\'run\',\'--task\',str(TASK)]\n    with (META/\'generation_stdout.log\').open(\'xb\') as outlog, (META/\'generation_stderr.log\').open(\'xb\') as errlog:\n        GENERATION=subprocess.Popen(command,stdout=outlog,stderr=errlog,start_new_session=True)\n        last_progress=0\n        while GENERATION.poll() is None:\n            now=time.monotonic()\n            if now-last_progress>=30:\n                occupied=len(list((RUN/\'cases\').glob(\'*/intent.json\'))) if (RUN/\'cases\').is_dir() else 0\n                saved=len(list((RUN/\'cases\').glob(\'*/http_response.bin\'))) if (RUN/\'cases\').is_dir() else 0\n                print(f\'Development progress: {occupied}/300 durable intents; {saved} saved HTTP responses. These are not correctness counts.\',flush=True)\n                last_progress=now\n            time.sleep(2)\n        generation_returncode=GENERATION.returncode\n    generation_stdout=(META/\'generation_stdout.log\').read_text(errors=\'replace\')\n    generation_stderr=(META/\'generation_stderr.log\').read_text(errors=\'replace\')\n    write(\'generation_command_output.json\',{\'command\':command,\'returncode\':generation_returncode,\'stdout\':generation_stdout,\'stderr\':generation_stderr})\n    if generation_returncode:raise RuntimeError(\'Generation failed; preserve durable request states; never rerun unknown requests.\')\n    print(generation_stdout,flush=True)\n    command=[sys.executable,str(TASK/\'repeated_client.py\'),\'replay\',\'--task\',str(TASK)]\n    completed=subprocess.run(command,capture_output=True,text=True)\n    write(\'replay_command_output.json\',{\'returncode\':completed.returncode,\'stdout\':completed.stdout,\'stderr\':completed.stderr})\n    if completed.returncode:raise RuntimeError(\'CPU replay failed; preserve outputs.\')\n    print(completed.stdout,flush=True)\n    write(\'execution_status.json\',{\'status\':\'generation_and_replay_finished\',\'held_out_access_enabled\':False,\'official_scoring\':\'not_performed\'})\nexcept BaseException as exc:\n    failure=type(exc).__name__+\': \'+str(exc)\n    write(\'execution_error.json\',{\'error\':failure,\'action\':\'Preserve evidence; do not retry unknown outcomes.\'})\n    print(\'STOPPED:\',failure,flush=True)\nfinally:\n    try:stop()\n    except Exception as exc:print(\'Shutdown check failed:\',type(exc).__name__,flush=True)\n    # Package partial evidence too, so failure never requires rerunning a request.\n    if (TASK/\'runner.py\').is_file():shutil.copyfile(TASK/\'runner.py\',META/\'execution_source.py\')\n    if (TASK/\'repeated_client.py\').is_file():shutil.copyfile(TASK/\'repeated_client.py\',META/\'repeated_client.py\')\n    for n in (\'environment_install.log\',\'dependency_check.log\'):\n        if (TASK/n).is_file():shutil.copyfile(TASK/n,META/n)\n    sources={\'prepared\':PREP,\'metadata\':META}\n    if \'RUN\' in globals() and RUN.is_dir():sources[\'run\']=RUN\n    if \'REPLAY\' in globals() and REPLAY.is_dir():sources[\'replay\']=REPLAY\n    contents={}\n    for prefix,folder in sources.items():\n        for p in sorted(folder.rglob(\'*\')):\n            if p.is_symlink():raise RuntimeError(\'Symlink evidence rejected.\')\n            if p.is_file():contents[prefix+\'/\'+p.relative_to(folder).as_posix()]=p.read_bytes()\n    hashes={n:sha(raw) for n,raw in contents.items()}\n    export=Path(\'/content\')/(\'uqroute_qwen7b_corrected_repeated_\'+uuid.uuid4().hex[:8]+\'.zip\')\n    with zipfile.ZipFile(export,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for n,raw in contents.items():z.writestr(n,raw)\n        z.writestr(\'metadata/export_file_hashes.json\',json.dumps(hashes,sort_keys=True,indent=2)+\'\\n\')\n    with zipfile.ZipFile(export) as z:\n        if z.testzip() is not None or any(sha(z.read(n))!=h for n,h in hashes.items()):raise RuntimeError(\'Export integrity failed.\')\n    with (TASK/\'export_path.txt\').open(\'x\',encoding=\'utf-8\') as f:f.write(str(export))\n    print(\'Archive:\',export,flush=True);print(\'SHA256:\',sha(export.read_bytes()),flush=True)\n    print(\'Evidence files:\',len(hashes),flush=True)\n    print(\'Server shutdown attempted; download the archive before disconnecting.\',flush=True)\nif failure:sys.exit(1)\n'
(TASK / 'runner.py').write_text(RUNNER_SOURCE, encoding='utf-8')
print('Starting validation, model loading, generation and replay.', flush=True)
execution = subprocess.run([str(PYTHON), str(TASK / 'runner.py'), str(TASK)], env=env)
print('Execution exit code:', execution.returncode)
if (TASK / 'export_path.txt').is_file():
    print('Evidence export ready. Run the download cell, including after a recorded run failure.')
else:
    print('No export produced. Preserve the runtime and logs; do not rerun generation.')


## Download evidence
Run after the execution cell finishes, including a recorded failure. Downloading again is safe; generation must not be repeated.


In [ ]:
from pathlib import Path
from google.colab import files
export_marker = Path('/content/uqroute_qwen7b_repeated_execution_v1/export_path.txt')
if not export_marker.is_file():
    raise RuntimeError('No export marker. Preserve the runtime and logs; do not rerun generation.')
export_path = Path(export_marker.read_text().strip())
if not export_path.is_file():
    raise RuntimeError('Export missing; preserve runtime.')
print('Downloading:', export_path.name)
files.download(str(export_path))
